<div align="center">

# Week 13: Human-AI Interaction I: Trust, Reliance and Explanations

**Human-Computer Interaction (BIL-429), hands-on Colab notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Week overview](https://github.com/utkukose/human-computer-interaction-NB-lecture/tree/main/weeks/week-13) · [Lecture page](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-13/lecture.html) · [Interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-13/lab.html) · [PDF notes](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-13/Week13_Lecture_Notes.pdf)

</div>

## How to use this Colab notebook

This Colab notebook is the hands-on part of the week. The concepts are explained on the [lecture page](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-13/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Cells titled *Observe and reflect* ask open questions about what the code shows. They have no solution cell, and a short answer in a text cell of your copy (Insert, Text cell) is enough. Cells titled *Python note* explain a piece of the Python language used nearby. Nothing needs to be installed.

**Starting in the middle.** Each numbered section can also be opened on its own, for example from an orange Colab box of the lecture page. Its first code cell, *Catch-up*, runs the code of the earlier sections without their printouts and figures. When the Colab notebook runs from the top, the catch-up cells notice it and do nothing.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback. It also stores the six departments and the twelve complaints of the triage form. For each department, it holds the share of each complaint, the ages and the share of the patients. The second cell keeps the expected values of the exercises in the table `ANSWERS`, with its code hidden, so that no answer stands next to its exercise.

In [ ]:
# Environment: Everything used by this course is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import io
import math
import random
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import optimize
from sklearn.linear_model import LogisticRegression

RANDOM_STATE = 42
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (7, 3.6),
                     "axes.grid": True, "grid.alpha": 0.3})

def check(name, value, expected=None, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback.
    Without an expected value in the call, check takes the value and its tolerance
    from ANSWERS, the table of the next cell."""
    if expected is None:
        answer, tol = ANSWERS[name]
        expected = answer()
    if value is None:
        print(f"{name}: not answered yet. Replace None and run the cell again.")
        return False
    try:
        scale = max(1.0, abs(float(expected)))
        ok = abs(float(value) - float(expected)) <= tol * scale
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else
                         "not yet. Compare your steps with the lecture."))
    return ok


# The six departments and the twelve complaints of the triage form
DEPTS = ["Cardiology", "Chest diseases", "Gastroenterology", "Neurology",
         "Orthopedics", "Dermatology"]
SYMPTOMS = ["chest pain", "palpitations", "shortness of breath", "cough",
            "fever", "abdominal pain", "nausea", "headache", "dizziness",
            "joint pain", "back pain", "rash"]
# Share of the patients of each department (one row) who report each complaint
P_SYM = np.array([
    [.60, .50, .45, .12, .06, .08, .20, .12, .40, .06, .10, .02],
    [.40, .10, .65, .70, .45, .05, .10, .15, .10, .06, .15, .03],
    [.15, .05, .06, .08, .30, .75, .65, .10, .10, .05, .20, .04],
    [.05, .10, .05, .04, .06, .05, .35, .70, .55, .05, .15, .03],
    [.05, .03, .04, .04, .04, .05, .04, .06, .06, .65, .60, .04],
    [.03, .03, .03, .04, .25, .04, .06, .06, .04, .25, .04, .85]])
AGE_MEAN = [64, 56, 47, 50, 45, 36]   # mean age of each department (years)
AGE_SD = [12, 15, 16, 17, 17, 15]     # spread of the age (years)
PRIOR = [0.20, 0.15, 0.20, 0.15, 0.20, 0.10]   # share of each department
print("Setup complete. Running in Colab:", IN_COLAB, "| departments:", len(DEPTS),
      "| complaints:", len(SYMPTOMS))

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
ANSWERS = {
    "Exercise 13.1": (lambda: 0.5, 1e-6),
    "Exercise 13.2": (lambda: 1 / (1 + math.exp(-1.5)), 1e-4),
    "Exercise 13.3": (lambda: math.exp(1.2) / (math.exp(1.2) + 2 * math.exp(0.3)), 1e-4),
    "Exercise A": (lambda: 0.9018, 0.003),
    "Exercise B": (lambda: 0.6 ** 2 + 0.7 ** 2 + 0.1 ** 2, 1e-6),
    "Exercise C": (lambda: 2.07 - 1.01, 0.011),
    "Exercise D": (lambda: 1 - (10 - 2) / 20, 1e-6),
    "Exercise E": (lambda: 0.85, 1e-6),
}

## Python step 13: Probabilities from counts and scores

An assistant of artificial intelligence (AI) that suggests a department gives a probability for each department. An honest display of these probabilities rests on counting how often the assistant is right. This step computes shares from arrays of zeros and ones and from comparisons. NumPy turns a condition into an array of True and False &#91;[1](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-13/lecture.html#ref-1)&#93;. It then turns a score into a probability with the logistic function and several scores into probabilities that add up to 1 with the softmax function. Finally, it does the same for many patients at once, one row per patient. The sections of the week use these pieces for the confidence of the assistant, for calibration and for explanations.

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "PRIOR" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-13/"
               "NB13_trust_reliance_explanations.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "python-step":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

### Shares from an array of zeros and ones

In [ ]:
# Python practice: a small example for learning the language
right = np.array([1, 1, 0, 1, 1, 1, 0, 1, 1, 1])    # was the suggestion right?
print("share right:", right.mean())
conf = np.array([0.95, 0.9, 0.55, 0.8, 0.97, 0.7, 0.6, 0.85, 0.9, 0.75])
sure = conf >= 0.8                 # an array of True and False
print(sure)
print("right among the sure ones:", right[sure].mean().round(3))

The mean of zeros and ones is the share of ones, here 8 of 10 suggestions. A comparison such as `conf >= 0.8` compares every element and returns an array of `True` and `False`. Placed in square brackets, it keeps only the elements where it is `True`: The six suggestions with a confidence of at least 0.8 were all right, a share of 1.0.

### A score becomes a probability: The logistic function

In [ ]:
# Python practice: a small example for learning the language
def logistic(z):
    """Turns a score into a probability between 0 and 1."""
    return 1 / (1 + np.exp(-z))

for z in (-2, 0, 1, 3):
    p = logistic(z)
    print(f"score {z:>2}: probability {p:.3f}, odds {p / (1 - p):.3f}")

The logistic function $p = \frac{1}{1 + e^{-z}}$ turns any score $z$ into a probability between 0 and 1. A score of 0 gives 0.5, and every point of score multiplies the *odds* $\frac{p}{1 - p}$ by $e \approx 2.718$: The odds equal $e^z$. The score is therefore the logarithm of the odds, and `np.exp` computes $e^z$ for a number or for a whole array.

### Several scores become probabilities: The softmax function

In [ ]:
# Python practice: a small example for learning the language
def softmax(z):
    """Scores of the classes -> probabilities that add up to 1."""
    e = np.exp(z - np.max(z))
    return e / e.sum()

scores = np.array([2.0, 1.0, -0.5])     # three departments
p = softmax(scores)
print(p.round(3), "| sum:", p.sum().round(3))
print("largest at position", np.argmax(p), "with", p.max().round(3))

The softmax function $p_k = \frac{e^{z_k}}{\sum_j e^{z_j}}$ makes every probability positive and scales them so that they add up to 1. With two classes and the scores $z$ and 0 it is the logistic function again. Subtracting the largest score first keeps `np.exp` away from huge numbers and does not change the result. `np.argmax` returns the position of the largest value, counted from 0.

### Many patients at once: Rows and the axis

In [ ]:
# Python practice: a small example for learning the language
Z = np.array([[2.0, 1.0, -0.5],
              [0.2, 0.4, 0.1]])          # one row of scores per patient
E = np.exp(Z - Z.max(axis=1, keepdims=True))
Pr = E / E.sum(axis=1, keepdims=True)
print(Pr.round(3))
print("suggestions:", Pr.argmax(axis=1), "| confidence:", Pr.max(axis=1).round(3))

With `axis=1`, `max`, `sum` and `argmax` work along every row, one result per patient. `keepdims=True` keeps the results as a column, so that each row is divided by its own sum. The second patient has close scores, so the largest probability, the *confidence* of the suggestion, is only 0.391.

**Quick check.** Why does the softmax not change when the same number is subtracted from every score?

<details><summary>Answer</summary>

Because $e^{z_k - c} = e^{z_k}\,e^{-c}$: The factor $e^{-c}$ appears in every term of the numerator and of the sum, and it cancels.

</details>

### Exercises for Python step 13

**Exercise 13.1.** Store the share of right suggestions among those with a confidence below 0.8 in `share_unsure`, with `right` and `conf` of the first cell.

In [ ]:
share_unsure = None   # your answer

check("Exercise 13.1", share_unsure)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
share_unsure = right[conf < 0.8].mean()

check("Exercise 13.1", share_unsure)

**Exercise 13.2.** Store the probability that the logistic function gives to the score 1.5 in `p_score`.

In [ ]:
p_score = None   # your answer

check("Exercise 13.2", p_score)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
p_score = logistic(1.5)

check("Exercise 13.2", p_score)

**Exercise 13.3.** Three departments have the scores 1.2, 0.3 and 0.3. Store the confidence of the suggestion, the largest probability of the softmax, in `top`.

In [ ]:
top = None   # your answer

check("Exercise 13.3", top)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
top = softmax(np.array([1.2, 0.3, 0.3])).max()

check("Exercise 13.3", top)

## 1. The assistant and the reliance on it &#91;[2](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-13/lecture.html#ref-2)&#93;

*Lecture: [Trust and reliance](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-13/lecture.html#trust-and-reliance)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "softmax" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-13/"
               "NB13_trust_reliance_explanations.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-1":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The assistant of this week suggests a department from the complaints and the age of a patient. Python generates synthetic patients, trains the assistant on them and measures its accuracy on new patients. It then computes how accurate the nurse and the assistant are together when the nurse checks only the uncertain suggestions, and it analyses your decisions of Part A.

**From the formulas to the code.** `make_patients(n, seed)` draws the department of each patient with the shares `PRIOR`, each complaint with the share in `P_SYM` and the age around `AGE_MEAN`. The assistant is a logistic regression, and its confidence $c$ is its largest probability. `team_accuracy(tau, h)` computes $A = s\,a_{\text{hi}} + (1 - s)\,h$, where $s$ is the share of suggestions with $c \ge \tau$, $a_{\text{hi}}$ is their accuracy and $h$ is the accuracy of the nurse.

In [ ]:
def make_patients(n, seed):
    """Synthetic patients: complaints (0 or 1), age (years), department."""
    rng = np.random.default_rng(seed)
    y = rng.choice(6, size=n, p=PRIOR)
    X = (rng.random((n, 12)) < P_SYM[y]).astype(int)
    age = rng.normal(np.array(AGE_MEAN)[y], np.array(AGE_SD)[y])
    age = np.clip(np.round(age), 18, 90).astype(int)
    return X, age, y

def features(X, age):
    """The twelve complaints and the age in decades from 50 years."""
    return np.column_stack([X, (age - 50) / 10])

X_tr, age_tr, y_tr = make_patients(1500, 1301)   # to train the assistant
X_ca, age_ca, y_ca = make_patients(500, 1302)    # to calibrate it, Section 2
X_te, age_te, y_te = make_patients(1000, 1303)   # new patients to test it
assistant = LogisticRegression(max_iter=5000, tol=1e-10)
assistant.fit(features(X_tr, age_tr), y_tr)
P = assistant.predict_proba(features(X_te, age_te))
pred, conf = P.argmax(axis=1), P.max(axis=1)
right = pred == y_te
print("accuracy on 1,000 new patients:", right.mean().round(3),
      "| mean confidence:", conf.mean().round(3))
for k, name in enumerate(DEPTS):
    mine = y_te == k
    print(f"  {name}: {mine.sum()} patients, suggested right "
          f"{right[mine].mean():.3f}")

**Python note.** `LogisticRegression` of scikit-learn &#91;[5](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-13/lecture.html#ref-5)&#93; learns one weight per feature and department. `fit` learns from patients whose department is known, and `predict_proba` returns one row of six probabilities per new patient. `argmax(axis=1)` and `max(axis=1)` come from Python step 13.

In [ ]:
NURSE = 0.9   # accuracy of the triage nurse who decides alone

def team_accuracy(tau, h=NURSE):
    """Accept the suggestions with conf >= tau; the nurse decides the rest."""
    hi = conf >= tau
    s = hi.mean()
    a_hi = right[hi].mean() if hi.any() else 0.0
    return s * a_hi + (1 - s) * h, s, a_hi

print("assistant alone:", right.mean().round(3), "| nurse alone:", NURSE)
for tau in (0.5, 0.6, 0.7, 0.8, 0.9):
    acc, s, a_hi = team_accuracy(tau)
    print(f"tau {tau}: accepted {s:.3f}, right among them {a_hi:.3f}, "
          f"team {acc:.3f}, the nurse checks {1 - s:.3f}")

**Python note.** `team_accuracy` returns three values at once, and `acc, s, a_hi = ...` unpacks them. `right[hi].mean() if hi.any() else 0.0` avoids the mean of an empty array when no suggestion reaches the threshold.

**Exercise A.** Store the team accuracy at the threshold 0.8 for a nurse who is right 85 percent of the time in `acc_team`.

In [ ]:
acc_team = None   # your answer

check("Exercise A", acc_team)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
acc_team = team_accuracy(0.8, 0.85)[0]

check("Exercise A", acc_team)

In [ ]:
# Figure: the accuracy of the team against the threshold, and the workload
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
taus = np.linspace(0.3, 0.99, 70)
for h, col in ((0.8, "#C98510"), (0.9, "#5A4BB5")):
    ax[0].plot(taus, [team_accuracy(t, h)[0] for t in taus], color=col,
               label=f"team, nurse {h}")
    ax[0].axhline(h, color=col, ls=":", lw=1)
ax[0].axhline(right.mean(), color="#B53F2E", ls="--", lw=1, label="assistant alone")
ax[0].set_xlabel("threshold tau: accept the suggestion at or above it")
ax[0].set_ylabel("share of right departments")
ax[0].legend(fontsize=8, loc="upper left")
ax[1].plot(taus, [1 - team_accuracy(t)[1] for t in taus], color="#2E8B57")
ax[1].set_xlabel("threshold tau")
ax[1].set_ylabel("share that the nurse checks")
plt.tight_layout()
plt.show()

**Python note.** The dotted lines show each nurse alone. Where a solid line rises above both its dotted line and the dashed line, the team is more accurate than either member.

**Paste your results.** Copy the results of Part A of the interactive lab and paste them between the triple quotes of `MY_TRIAGE` below. If you leave it empty, the code uses the results of a typical student.

In [ ]:
MY_TRIAGE = """
"""
DEFAULT_TRIAGE = """
round,case,age,ai,conf,ai_ok,asked,choice,truth,correct,accepted,ms
1,1,55,Neurology,0.94,1,0,Neurology,Neurology,1,1,4512
1,2,44,Chest diseases,0.8,1,0,Chest diseases,Chest diseases,1,1,6574
1,3,70,Chest diseases,0.64,1,0,Chest diseases,Chest diseases,1,1,5552
1,4,46,Dermatology,0.93,1,1,Dermatology,Dermatology,1,1,10623
1,5,41,Gastroenterology,0.54,0,1,Orthopedics,Orthopedics,1,0,13585
1,6,40,Gastroenterology,0.91,1,1,Gastroenterology,Gastroenterology,1,1,11311
1,7,40,Dermatology,0.87,1,0,Dermatology,Dermatology,1,1,5046
1,8,57,Cardiology,0.84,1,0,Cardiology,Cardiology,1,1,3850
1,9,62,Cardiology,0.41,0,1,Orthopedics,Orthopedics,1,0,9329
1,10,69,Neurology,0.91,1,0,Neurology,Neurology,1,1,5588
1,11,45,Cardiology,0.47,0,0,Cardiology,Gastroenterology,0,1,6330
1,12,52,Cardiology,0.69,1,0,Cardiology,Cardiology,1,1,5504
2,1,32,Gastroenterology,0.99,1,0,Gastroenterology,Gastroenterology,1,1,2181
2,2,40,Dermatology,0.87,1,0,Dermatology,Dermatology,1,1,3713
2,3,55,Cardiology,0.78,1,0,Cardiology,Cardiology,1,1,4622
2,4,65,Chest diseases,0.99,1,0,Chest diseases,Chest diseases,1,1,3020
2,5,49,Gastroenterology,0.98,1,0,Gastroenterology,Gastroenterology,1,1,5027
2,6,56,Neurology,0.66,1,1,Neurology,Neurology,1,1,8735
2,7,74,Cardiology,0.95,1,0,Cardiology,Cardiology,1,1,3595
2,8,63,Gastroenterology,0.44,0,1,Orthopedics,Orthopedics,1,0,12250
2,9,90,Chest diseases,0.81,1,0,Chest diseases,Chest diseases,1,1,4863
2,10,34,Orthopedics,0.7,1,0,Orthopedics,Orthopedics,1,1,3447
2,11,48,Chest diseases,0.65,0,1,Neurology,Neurology,1,0,12097
2,12,24,Gastroenterology,0.59,0,1,Dermatology,Dermatology,1,0,10573"""
src = MY_TRIAGE.strip() or DEFAULT_TRIAGE.strip()
print("Data:", "your own decisions from the interactive lab" if MY_TRIAGE.strip()
      else "the results of a typical student")
tri = pd.read_csv(io.StringIO(src))
for rd, part in tri.groupby("round"):
    wrong, ok = part[part.ai_ok == 0], part[part.ai_ok == 1]
    print(f"Round {rd}: assistant right {len(ok)} of {len(part)}, team right "
          f"{part.correct.sum()} | accepted a wrong suggestion "
          f"{wrong.accepted.sum()} of {len(wrong)} | changed a right one "
          f"{len(ok) - ok.accepted.sum()} of {len(ok)} | asked {part.asked.sum()}"
          f" | mean time {part.ms.mean() / 1000:.1f} s")
asked = tri[tri.asked == 1]
print("Asked patients by the confidence of the suggestion (shown in round 2):")
print(asked[["round", "case", "conf", "ai_ok"]].to_string(index=False))

**Python note.** Accepting a wrong suggestion is overreliance, and changing a right one is underreliance. `len(ok) - ok.accepted.sum()` counts the right suggestions that were not accepted.

**Explore.** Move the sliders: The code prints the team accuracy and the share that the nurse checks. If the controls do not appear, run the cell again.

In [ ]:
import ipywidgets as widgets

def explore(tau=0.9, h=0.9):
    acc, s, a_hi = team_accuracy(tau, h)
    print(f"accepted {s:.3f} (right {a_hi:.3f}) | team {acc:.3f} | "
          f"the nurse checks {1 - s:.3f}")

widgets.interact(explore,
                 tau=widgets.FloatSlider(value=0.9, min=0.3, max=1.0, step=0.05),
                 h=widgets.FloatSlider(value=0.9, min=0.5, max=1.0, step=0.05));

**Observe and reflect.** Whom did you ask in round 1, and why? Did the confidence of round 2 move your questions to the suggestions that needed them?

## 2. Confidence and calibration &#91;[6](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-13/lecture.html#ref-6)&#93;

*Lecture: [Confidence and calibration](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-13/lecture.html#confidence-and-calibration)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "team_accuracy" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-13/"
               "NB13_trust_reliance_explanations.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-2":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A confidence display helps only when its numbers mean what they say. Python sorts the suggestions into ten bins of confidence and compares the confidence with the share right in each bin. It computes the expected calibration error (ECE) and the Brier score (BS) of the assistant, of an overconfident version and of that version after temperature scaling.

**From the formulas to the code.** `reliability(c, ok)` returns for each bin the count $n_b$, the mean confidence $\bar c_b$ and the share right $a_b$. `ece(c, ok)` computes $\mathrm{ECE} = \sum_b \frac{n_b}{N}\,|a_b - \bar c_b|$. `brier(P, y)` computes $\mathrm{BS} = \frac{1}{N} \sum_i \sum_k (p_{ik} - o_{ik})^2$, with $o_{ik} = 1$ for the right department and 0 otherwise. `soft(Z, T)` computes the softmax of the scores divided by the temperature $T$.

In [ ]:
def reliability(c, ok, bins=10):
    """Count, mean confidence and share right in each bin of confidence."""
    b = np.minimum((c * bins).astype(int), bins - 1)
    rows = []
    for k in range(bins):
        m = b == k
        if m.any():
            rows.append((f"{k / bins:.1f}-{(k + 1) / bins:.1f}", m.sum(),
                         round(c[m].mean(), 3), round(ok[m].mean(), 3)))
    return pd.DataFrame(rows, columns=["bin", "n", "confidence", "right"])

def ece(c, ok, bins=10):
    """Expected calibration error: the weighted gap of the bins."""
    t = reliability(c, ok, bins)
    return float((t.n / t.n.sum() * (t.right - t.confidence).abs()).sum())

def brier(P, y):
    """Mean over the patients of the squared differences over the classes."""
    O = np.eye(P.shape[1])[y]          # 1 for the right department, else 0
    return float(np.mean(np.sum((P - O) ** 2, axis=1)))

print(reliability(conf, right).to_string(index=False))
print("ECE:", round(ece(conf, right), 4), "| Brier score:", round(brier(P, y_te), 4))

**Python note.** `(c * bins).astype(int)` gives the number of the bin, and `np.minimum(..., bins - 1)` puts a confidence of exactly 1 into the last bin. `np.eye(6)[y]` picks one row of the identity matrix per patient: A row with 1 for the right department and 0 elsewhere.

**Exercise B.** The assistant gives Cardiology 0.6, Chest diseases 0.3 and Neurology 0.1, and 0 to the other three departments. The patient belonged to Chest diseases. Store the Brier score of this one suggestion in `bs`.

In [ ]:
bs = None   # your answer

check("Exercise B", bs)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
bs = brier(np.array([[0.6, 0.3, 0, 0.1, 0, 0]]), np.array([1]))

check("Exercise B", bs)

In [ ]:
def soft(Z, T=1.0):
    """Softmax of every row of scores Z, divided by the temperature T."""
    E = np.exp((Z - Z.max(axis=1, keepdims=True)) / T)
    return E / E.sum(axis=1, keepdims=True)

Z_te = assistant.decision_function(features(X_te, age_te))   # the scores
Z_ca = assistant.decision_function(features(X_ca, age_ca))
VENDOR_T = 0.5          # the vendor sharpens every suggestion
vendor = soft(Z_te, VENDOR_T)
print("vendor: mean confidence", vendor.max(axis=1).mean().round(3),
      "| ECE", round(ece(vendor.max(axis=1), right), 4),
      "| Brier", round(brier(vendor, y_te), 4))

def nll(T, Z, y):
    """Mean negative log-probability of the right department."""
    return -np.mean(np.log(soft(Z, T)[np.arange(len(y)), y]))

res = optimize.minimize_scalar(nll, bounds=(0.2, 5), method="bounded",
                               args=(Z_ca / VENDOR_T, y_ca))
T_FIT = round(res.x, 3)
fixed = soft(Z_te / VENDOR_T, T_FIT)
print(f"fitted temperature {T_FIT} | after scaling: mean confidence "
      f"{fixed.max(axis=1).mean():.3f}, ECE {ece(fixed.max(axis=1), right):.4f}, "
      f"Brier {brier(fixed, y_te):.4f}")
print("suggestions changed by the scaling:",
      int((fixed.argmax(axis=1) != pred).sum()))

**Python note.** `decision_function` returns the six scores of each patient before the softmax. `minimize_scalar` of SciPy &#91;[8](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-13/lecture.html#ref-8)&#93; finds the temperature with the smallest negative log-probability on the calibration patients. Guo and colleagues fitted the temperature in the same way &#91;[6](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-13/lecture.html#ref-6)&#93;. `soft(Z, T)[np.arange(n), y]` picks the probability of the right department in every row.

In [ ]:
# Figure: reliability diagrams of the assistant, the vendor and the scaled vendor
fig, ax = plt.subplots(1, 3, figsize=(11, 3.5), sharey=True)
for a, (name, PP) in zip(ax, (("assistant", P), ("vendor", vendor),
                              ("vendor after scaling", fixed))):
    c = PP.max(axis=1)
    t = reliability(c, right)
    mids = t.confidence
    a.bar(mids, t.right, width=0.07, color="#5A4BB5", alpha=0.8, label="share right")
    a.plot([0, 1], [0, 1], color="#585E6E", ls="--", lw=1, label="perfect")
    a.scatter(mids, t.right, s=t.n / 4, color="#1E2433", zorder=3)
    a.set_title(f"{name}: ECE {ece(c, right):.3f}", fontsize=9)
    a.set_xlabel("confidence of the suggestion")
ax[0].set_ylabel("share right")
ax[0].legend(fontsize=7, loc="upper left")
plt.tight_layout()
plt.show()

**Python note.** Each bar stands at the mean confidence of its bin and reaches the share right. Bars below the dashed line mean overconfidence, and the size of each dot shows how many suggestions the bin holds.

**Explore.** Move the slider: The code divides the vendor's scores by another temperature and prints the mean confidence, the expected calibration error and the Brier score. If the controls do not appear, run the cell again.

In [ ]:
import ipywidgets as widgets

def explore(T=1.0):
    PP = soft(Z_te / VENDOR_T, T)
    c = PP.max(axis=1)
    print(f"T {T:.2f}: mean confidence {c.mean():.3f} | accuracy "
          f"{right.mean():.3f} | ECE {ece(c, right):.4f} | Brier "
          f"{brier(PP, y_te):.4f}")

widgets.interact(explore, T=widgets.FloatSlider(value=1.0, min=0.3, max=4.0,
                                                step=0.1));

**Observe and reflect.** The vendor's version and the scaled version suggest the same departments. What would a nurse learn about each of them after a month of use?

## 3. Explanations &#91;[10](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-13/lecture.html#ref-10)&#93;

*Lecture: [Explanations: Why this department and not another?](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-13/lecture.html#explanations)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "brier" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-13/"
               "NB13_trust_reliance_explanations.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-3":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** An explanation should help a person predict what the assistant will do. Python computes the points that each complaint and the age add to the lead of a suggestion over the runner-up and checks that they add up to the lead. It predicts what happens when a complaint is removed, and it analyses your predictions of Part C.

**From the formulas to the code.** `contrast(x, age, a, b)` computes $\Delta_j = (w_{aj} - w_{bj})\,x_j$ for the complaints present and for the age, sorted by size, and the lead $z_a - z_b$. The lead equals $\Delta_0 + \sum_j \Delta_j$, where $\Delta_0 = w_{a0} - w_{b0}$ is the difference of the intercepts. Removing a complaint lowers the lead by its $\Delta_j$.

In [ ]:
Wt, w0 = assistant.coef_, assistant.intercept_   # 6 x 13 weights, 6 intercepts

def contrast(x, age, a, b):
    """Points that each present feature adds to the lead of a over b."""
    f = np.append(x, (age - 50) / 10)
    names = SYMPTOMS + [f"age {age}"]
    d = (Wt[a] - Wt[b]) * f
    items = [(names[j], round(d[j], 2)) for j in range(13) if f[j] != 0]
    items.sort(key=lambda t: -abs(t[1]))
    lead = round(float(f @ (Wt[a] - Wt[b]) + w0[a] - w0[b]), 2)
    return items, lead, round(float(w0[a] - w0[b]), 2)

def suggest(x, age):
    """Probabilities of the six departments for one patient."""
    return assistant.predict_proba(features(np.array([x]), np.array([age])))[0]

x = np.isin(SYMPTOMS, ["chest pain", "shortness of breath", "cough"]).astype(int)
p = suggest(x, 58)
a, b = np.argsort(-p)[:2]
print(f"suggestion {DEPTS[a]} {p[a]:.3f}, runner-up {DEPTS[b]} {p[b]:.3f}")
items, lead, base = contrast(x, 58, a, b)
for name, d in items:
    print(f"  {name}: {d:+.2f} points")
total = round(base + sum(d for _, d in items), 2)
print("intercepts:", f"{base:+.2f}", "| sum:", total, "| lead:", lead,
      "| log of the ratio:", round(math.log(p[a] / p[b]), 3))

**Python note.** `np.isin(SYMPTOMS, [...])` marks the complaints of the list with `True`, and `astype(int)` turns them into ones. `np.argsort(-p)` sorts the positions from the largest probability down. `f @ v` is the dot product of Week 12. The lead is the logarithm of the ratio of the two probabilities.

**Exercise C.** The patient of the example says that the shortness of breath was a mistake. Store the new lead of Chest diseases over Cardiology, the lead of the example minus the points of the shortness of breath, in `lead2`.

In [ ]:
lead2 = None   # your answer

check("Exercise C", lead2)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
lead2 = lead - dict(items)["shortness of breath"]

check("Exercise C", lead2)

In [ ]:
for name, d in items:
    if name.startswith("age"):
        continue
    x2 = x.copy()
    x2[SYMPTOMS.index(name)] = 0
    p2 = suggest(x2, 58)
    k = p2.argmax()
    rule = DEPTS[b] if d > lead else DEPTS[a]
    print(f"without {name}: lead {lead - d:+.2f} -> the rule says {rule}, "
          f"the assistant says {DEPTS[k]} {p2[k]:.3f}")

**Python note.** `x.copy()` makes a new array, so that removing a complaint does not change `x`. The rule of the explanation predicts a switch to the runner-up when the points of the complaint exceed the lead.

**Paste your results.** Copy the results of Part C of the interactive lab and paste them between the triple quotes of `MY_WHATIF` below. The departments appear as their numbers in `DEPTS`, from 0 for Cardiology to 5 for Dermatology. If you leave it empty, the code uses the results of a typical student.

In [ ]:
MY_WHATIF = """
"""
DEFAULT_WHATIF = """
round,trial,ai,runner,removed,delta,lead,pred,new,correct,ms
1,1,2,0,abdominal pain,3.27,0.89,0,0,1,5067
1,2,5,2,joint pain,0.79,2.31,2,5,0,7302
1,3,4,3,joint pain,3.23,1.79,4,3,0,7832
1,4,3,2,dizziness,2.37,0.99,3,2,0,10437
1,5,0,1,chest pain,0.61,1.2,0,0,1,7707
1,6,1,0,shortness of breath,1.01,1.42,0,1,0,5451
1,7,0,1,nausea,0.22,0.53,0,0,1,9423
1,8,2,3,abdominal pain,3.25,1.09,2,3,0,7497
2,1,0,3,chest pain,2.92,1.45,3,3,1,7369
2,2,4,2,joint pain,2.7,0.94,2,2,1,11728
2,3,3,0,dizziness,0.91,1.82,3,3,1,9343
2,4,0,1,chest pain,0.61,1.15,0,0,1,9255
2,5,1,0,shortness of breath,1.01,1.4,0,1,0,12390
2,6,2,4,abdominal pain,4.07,0.97,4,4,1,8302
2,7,1,0,cough,2.8,0.43,0,0,1,7421
2,8,3,0,dizziness,0.91,1.49,3,3,1,11860"""
src = MY_WHATIF.strip() or DEFAULT_WHATIF.strip()
print("Data:", "your own predictions from the interactive lab" if MY_WHATIF.strip()
      else "the results of a typical student")
wif = pd.read_csv(io.StringIO(src))
wif["kind"] = np.where(wif.new == wif.ai, "stay", "switch")
wif["rule"] = np.where(wif.delta > wif.lead, wif.runner, wif.ai)
for rd, part in wif.groupby("round"):
    label = "without the explanation" if rd == 1 else "with the explanation"
    st, sw = part[part.kind == "stay"], part[part.kind == "switch"]
    print(f"Round {rd}, {label}: predicted {part.correct.sum()} of {len(part)} | "
          f"stays {st.correct.sum()} of {len(st)}, switches {sw.correct.sum()} "
          f"of {len(sw)} | mean time {part.ms.mean() / 1000:.1f} s")
print("The rule of the explanation is right in", (wif.rule == wif.new).sum(),
      "of", len(wif), "trials")

**Python note.** `np.where(test, a, b)` takes `a` where the test holds and `b` elsewhere, element by element. The rule uses only the bar of the removed complaint and the lead, as the explanation of round 2 shows them.

In [ ]:
# Figure: the explanation of the example, and the suggestion without each complaint
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
names = [n for n, _ in items][::-1]
vals = [d for _, d in items][::-1]
ax[0].barh(names, vals, color=["#2E8B57" if v > 0 else "#B53F2E" for v in vals])
ax[0].axvline(0, color="#1E2433", lw=0.8)
ax[0].set_xlabel(f"points for {DEPTS[a]} rather than {DEPTS[b]}")
ax[0].set_title(f"lead {lead:+.2f} = intercepts {base:+.2f} + bars", fontsize=9)
labels, tops = ["all three"], [p]
for name in ["cough", "shortness of breath", "chest pain"]:
    x2 = x.copy()
    x2[SYMPTOMS.index(name)] = 0
    labels.append(f"no {name}")
    tops.append(suggest(x2, 58))
tops = np.array(tops)
for k, col in ((a, "#5A4BB5"), (b, "#C98510")):
    ax[1].plot(labels, tops[:, k], "o-", color=col, label=DEPTS[k])
ax[1].set_ylabel("probability")
ax[1].tick_params(axis="x", labelsize=7)
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Python note.** Green bars push towards the suggestion and red bars towards the runner-up. `[::-1]` reverses a list, so that the largest bar appears at the top of `barh`.

**Observe and reflect.** In which trials of Part C did the explanation change your prediction? Was there a trial where you would have liked to see a bar that the explanation left out?

## 4. Trust over time: The delivery robot &#91;[13](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-13/lecture.html#ref-13)&#93;

*Lecture: [Trust over time: Aversion and appreciation](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-13/lecture.html#trust-over-time)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "contrast" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-13/"
               "NB13_trust_reliance_explanations.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-4":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Trust changes with every outcome that a person sees, and only with the outcomes that the person sees. Python computes the break-even reliability of the delivery robot of Part D and the expected minutes of simple policies. It runs the trust model of the lecture with and without the display and compares the minutes with your trips.

**From the formulas to the code.** `break_even(t_s, t_w, t_f)` computes $p^* = 1 - \frac{t_w - t_s}{t_f}$. `expected_minutes(sends)` adds $t_s + (1 - p)\,t_f$ for each trip with the robot and $t_w$ for each walk, with the reliability $p$ of the phase. `trust_run` sends the robot when the trust $\theta$ exceeds $p^*$. After a success $\theta \leftarrow \theta + \alpha_s(1 - \theta)$, after a failure $\theta \leftarrow \theta - \alpha_f\,\theta$, and with the display $\theta \leftarrow \theta + \beta\,(r - \theta)$ before each trip.

In [ ]:
SEND, WALK, FAIL = 2, 8, 20   # minutes: robot, walking, extra after a failure
PHASES = [0.95, 0.95, 0.95, 0.95, 0.95, 0.95,
          0.5, 0.5, 0.5, 0.5, 0.5, 0.5,
          0.9, 0.9, 0.9, 0.9, 0.9, 0.9]   # reliability of each trip
OUTCOMES = {   # 1: the robot delivers, 0: it fails; one list per round
    1: [1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1],
    2: [1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1]}
RATES = [0.9, 0.9, 1.0, 1.0, 0.95, 0.95,
         0.9, 0.75, 0.65, 0.55, 0.7, 0.65,
         0.5, 0.75, 0.95, 0.9, 0.85, 0.95]   # display of round 2

def break_even(t_s=SEND, t_w=WALK, t_f=FAIL):
    """Reliability above which sending the robot saves time."""
    return round(1 - (t_w - t_s) / t_f, 3)

def expected_minutes(sends):
    """Expected minutes of the 18 trips for a list of 0 (walk) and 1 (send)."""
    return round(float(sum(SEND + (1 - p) * FAIL if s else WALK
                           for s, p in zip(sends, PHASES))), 1)

P_STAR = break_even()
oracle = [int(p > P_STAR) for p in PHASES]
print("break-even reliability:", P_STAR)
for name, sends in (("always send", [1] * 18), ("always walk", [0] * 18),
                    ("best policy", oracle)):
    print(f"{name}: {expected_minutes(sends)} expected minutes")

**Python note.** `int(p > P_STAR)` turns `True` into 1 and `False` into 0. The best policy knows the reliability of every phase, which no nurse knows in advance.

**Exercise D.** On a busy day, walking to the laboratory takes 10 minutes. Store the break-even reliability of the robot in `p_star`.

In [ ]:
p_star = None   # your answer

check("Exercise D", p_star)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
p_star = break_even(t_w=10)

check("Exercise D", p_star)

In [ ]:
TRUST0, A_S, A_F, BETA = 0.8, 0.1, 0.4, 0.5

def trust_run(outcomes, rates=None, t0=TRUST0, a_s=A_S, a_f=A_F, beta=BETA):
    """Send when trust > P_STAR; update the trust after each delivery seen."""
    trust, sends, trace = t0, [], []
    for t, ok in enumerate(outcomes):
        if rates is not None:
            trust += beta * (rates[t] - trust)       # the display
        trace.append(round(trust, 3))
        send = trust > P_STAR
        sends.append(int(send))
        if send:                                     # only sent trips are seen
            trust += a_s * (1 - trust) if ok else -a_f * trust
    return sends, trace

for rd in (1, 2):
    for shown in (None, RATES):
        sends, trace = trust_run(OUTCOMES[rd], shown)
        label = "with the display" if shown else "without it"
        print(f"outcomes of round {rd}, {label}: sends {sends}, "
              f"{expected_minutes(sends)} expected minutes")

**Python note.** `rates=None` makes the display optional: `if rates is not None` runs its update only when a list is given. Without the display, a walk teaches the model nothing, so its trust stays where it is.

**Paste your results.** Copy the results of Part D of the interactive lab and paste them between the triple quotes of `MY_ROBOT` below. If you leave it empty, the code uses the results of a typical student.

In [ ]:
MY_ROBOT = """
"""
DEFAULT_ROBOT = """
round,trip,p,choice,outcome,minutes,rate,ms
1,1,0.95,send,1,2,,4788
1,2,0.95,send,1,2,,2452
1,3,0.95,send,1,2,,2550
1,4,0.95,send,1,2,,4154
1,5,0.95,send,1,2,,4342
1,6,0.95,send,1,2,,3413
1,7,0.5,send,0,22,,4537
1,8,0.5,send,1,2,,1971
1,9,0.5,send,0,22,,1885
1,10,0.5,walk,,8,,2898
1,11,0.5,walk,,8,,4657
1,12,0.5,walk,,8,,5244
1,13,0.9,walk,,8,,2139
1,14,0.9,walk,,8,,3861
1,15,0.9,walk,,8,,4056
1,16,0.9,walk,,8,,3186
1,17,0.9,walk,,8,,3031
1,18,0.9,walk,,8,,4291
2,1,0.95,send,1,2,0.9,4635
2,2,0.95,send,1,2,0.9,5235
2,3,0.95,send,1,2,1.0,5555
2,4,0.95,send,1,2,1.0,4215
2,5,0.95,send,1,2,0.95,4276
2,6,0.95,send,1,2,0.95,4784
2,7,0.5,walk,,8,0.9,4601
2,8,0.5,walk,,8,0.75,5378
2,9,0.5,walk,,8,0.65,6402
2,10,0.5,walk,,8,0.55,4136
2,11,0.5,walk,,8,0.7,5099
2,12,0.5,walk,,8,0.65,4127
2,13,0.9,walk,,8,0.5,3375
2,14,0.9,send,1,2,0.75,5013
2,15,0.9,send,0,22,0.95,3713
2,16,0.9,send,1,2,0.9,5056
2,17,0.9,send,1,2,0.85,4526
2,18,0.9,send,1,2,0.95,4287"""
src = MY_ROBOT.strip() or DEFAULT_ROBOT.strip()
print("Data:", "your own trips from the interactive lab" if MY_ROBOT.strip()
      else "the results of a typical student")
rob = pd.read_csv(io.StringIO(src))
rob["phase"] = (rob.trip - 1) // 6 + 1
for rd, part in rob.groupby("round"):
    sends = (part.choice == "send").astype(int).tolist()
    per = part.groupby("phase").choice.apply(lambda c: (c == "send").sum()).tolist()
    print(f"Round {rd}: sent {per} of 6 per phase | {part.minutes.sum()} minutes, "
          f"{expected_minutes(sends)} expected (best {expected_minutes(oracle)})")

**Python note.** `(trip - 1) // 6 + 1` numbers the phases 1, 2 and 3, with `//` as the division without the remainder. The expected minutes remove the luck of the single failures, so the rounds can be compared.

In [ ]:
# Figure: the trust of the model against the reliability, and your trips
fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.5), sharey=True)
trips = np.arange(1, 19)
for a, rd in zip(ax, (1, 2)):
    a.step(trips, PHASES, where="mid", color="#585E6E", label="reliability")
    a.axhline(P_STAR, color="#C98510", ls=":", lw=1, label="break-even")
    for shown, col, lab in ((None, "#B53F2E", "trust, no display"),
                            (RATES, "#2E8B57", "trust, display")):
        a.plot(trips, trust_run(OUTCOMES[rd], shown)[1], "o-", ms=3, color=col,
               label=lab)
    part = rob[rob["round"] == rd]
    a.scatter(part.trip, np.where(part.choice == "send", 1.04, -0.04), marker="|",
              s=80, color="#1E2433", label="your sends (top)")
    a.set_title(f"outcomes of round {rd}", fontsize=9)
    a.set_xlabel("trip")
ax[0].set_ylabel("reliability and trust")
ax[0].legend(fontsize=7, loc="lower left")
plt.tight_layout()
plt.show()

**Python note.** `plt.step` draws the reliability as steps, one level per phase. Your sends appear as marks along the top and your walks along the bottom.

**Explore.** Move the sliders: The code runs the trust model on the outcomes of round 2, with and without the display. If the controls do not appear, run the cell again.

In [ ]:
import ipywidgets as widgets

def explore(a_s=0.1, a_f=0.4, beta=0.5):
    for shown in (None, RATES):
        sends, trace = trust_run(OUTCOMES[2], shown, a_s=a_s, a_f=a_f, beta=beta)
        label = "display" if shown else "no display"
        print(f"{label}: sends {sum(sends)} of 18, "
              f"{expected_minutes(sends)} expected minutes")

widgets.interact(explore,
                 a_s=widgets.FloatSlider(value=0.1, min=0.02, max=0.5, step=0.02),
                 a_f=widgets.FloatSlider(value=0.4, min=0.02, max=0.8, step=0.02),
                 beta=widgets.FloatSlider(value=0.5, min=0.1, max=1.0, step=0.1));

**Observe and reflect.** After the failures of round 1, did you send the robot again before the end? What would have made you try it again?

## 5. The clinic app, stage 13

*Lecture: [The clinic app, stage 13: An AI assistant that suggests a department](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-13/lecture.html#the-clinic-app-stage-13)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "trust_run" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-13/"
               "NB13_trust_reliance_explanations.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-5":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Stage 13 adds the triage assistant to the clinic app. The app shows a suggestion with a calibrated confidence and a short contrastive explanation, and it refers the patient to the triage nurse when the confidence is below the threshold. Python sets the threshold, computes what patients and nurses can expect and collects the results of the week in one dictionary.

In [ ]:
TAU = 0.8    # below it, the app refers the patient to the nurse
STAGE13 = {}
acc, s, a_hi = team_accuracy(TAU)
STAGE13["accuracy of the assistant alone"] = round(float(right.mean()), 3)
STAGE13["suggestions shown at tau 0.8"] = round(float(s), 3)
STAGE13["right among the shown suggestions"] = round(float(a_hi), 3)
STAGE13["patients referred to the nurse per 100"] = round(100 * (1 - float(s)), 1)
STAGE13["team accuracy with the nurse"] = round(float(acc), 3)
STAGE13["ECE of the assistant"] = round(ece(conf, right), 4)
STAGE13["ECE of the vendor"] = round(ece(vendor.max(axis=1), right), 4)
STAGE13["fitted temperature"] = T_FIT
STAGE13["ECE of the vendor after scaling"] = round(ece(fixed.max(axis=1), right), 4)
STAGE13["expected minutes, best robot policy"] = expected_minutes(oracle)
for shown, key in ((None, "no display"), (RATES, "display")):
    STAGE13[f"expected minutes, trust model, {key}"] = expected_minutes(
        trust_run(OUTCOMES[2], shown)[0])
for key, value in STAGE13.items():
    print(f"{key}: {value}")

**Python note.** `float(...)` turns NumPy numbers into plain Python numbers, so that the dictionary prints cleanly. The values come from the functions of Sections 1 to 4.

**Exercise E.** Find the smallest threshold among 0.50, 0.55, ..., 0.95 at which at least 95 percent of the shown suggestions are right, and store it in `tau95`.

In [ ]:
tau95 = None   # your answer

check("Exercise E", tau95)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
tau95 = next(t for t in np.round(np.arange(0.5, 0.96, 0.05), 2)
             if team_accuracy(t)[2] >= 0.95)

check("Exercise E", tau95)

In [ ]:
# Figure: the threshold of stage 13, calibration and the delivery robot
fig, ax = plt.subplots(1, 3, figsize=(11, 3.3))
taus = np.linspace(0.3, 0.99, 70)
vals = np.array([team_accuracy(t) for t in taus])
ax[0].plot(taus, vals[:, 1], color="#2E8B57", label="shown")
ax[0].plot(taus, vals[:, 2], color="#5A4BB5", label="right among shown")
ax[0].plot(taus, vals[:, 0], color="#C98510", label="team with nurse")
ax[0].axvline(TAU, color="#1E2433", ls=":", lw=1)
ax[0].set_xlabel("threshold tau")
ax[0].legend(fontsize=7)
names = ["assistant", "vendor", "vendor\nscaled"]
ax[1].bar(names, [STAGE13["ECE of the assistant"], STAGE13["ECE of the vendor"],
                  STAGE13["ECE of the vendor after scaling"]],
          color=["#5A4BB5", "#B53F2E", "#2E8B57"])
ax[1].set_ylabel("expected calibration error")
mins = [STAGE13["expected minutes, best robot policy"],
        STAGE13["expected minutes, trust model, no display"],
        STAGE13["expected minutes, trust model, display"]]
ax[2].bar(["best", "no display", "display"], mins,
          color=["#585E6E", "#B53F2E", "#2E8B57"])
ax[2].set_ylabel("expected minutes of 18 trips")
plt.tight_layout()
plt.show()

**Python note.** The three panels follow the three parts of stage 13: When to show a suggestion, how honest its confidence is and how trust follows the reliability of the robot.

## 6. Your application: The application switch

*Lecture: [Your application](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-13/lecture.html#your-application)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "STAGE13" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-13/"
               "NB13_trust_reliance_explanations.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-6":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The model of reliance applies to any application with an AI suggestion. Each application below has suggestions above and below a threshold of confidence and a person who can decide alone.

The values are illustrative. Set `APP` to one of the five names and run the cell.

In [ ]:
APP = "loan_review"
# clinic, spam_filter, loan_review, code_assistant, photo_tagging
APPS = {   # accuracy above and below the threshold, share above it,
           # and the accuracy of the person who decides alone
    "clinic": (0.98, 0.69, 0.39, 0.9),
    "spam_filter": (0.995, 0.85, 0.8, 0.97),
    "loan_review": (0.95, 0.7, 0.5, 0.85),
    "code_assistant": (0.9, 0.55, 0.45, 0.8),
    "photo_tagging": (0.97, 0.75, 0.6, 0.99)}

a_hi, a_lo, s, h = APPS[APP]
alone = s * a_hi + (1 - s) * a_lo
team = s * a_hi + (1 - s) * h
print(f"{APP}: the AI alone {alone:.3f}, the person alone {h:.3f}, together "
      f"{team:.3f}; the person checks {1 - s:.3f} of the cases")
if team > max(alone, h):
    print("  The team is more accurate than either member.")
else:
    print("  The team is not more accurate than its best member.")

**Python note.** The same formula as `team_accuracy`, with the shares and accuracies given directly instead of computed from patients.

**Observe and reflect.** In your application, what does a person lose when the AI is wrong, and what does a check cost? Which of the two decides your threshold?

## Application challenges

Each challenge transfers the ideas of the week to a field of computing. Choose the one closest to your interests and solve it in a copy of this Colab notebook.

| Field | Challenge |
|---|---|
| Software engineering | Write a function that returns the contrastive explanation of a suggestion as one short sentence with its two largest bars, and test it on three patients. |
| Web development | A web page shows the confidence of a suggestion as a bar. Compute with `reliability` how wide the bar should be for each bin, so that it shows the share right instead of the raw confidence. |
| Mobile development | A phone shows only the suggestions with a confidence of at least 0.9. Compute with `team_accuracy` how many suggestions per 100 patients it would hide. |
| Game development | Give a companion character of a game a reliability that drops in one level. Find with `trust_run` the display rate that brings the trust of a player back fastest. |
| Embedded systems and the Internet of Things | A sensor with a battery reports its reliability once per hour. Change `trust_run` so that the display updates only every third trip, and compare the expected minutes. |
| Data science | Train the assistant on 300 instead of 1,500 patients, and compare its accuracy, its expected calibration error and its Brier score with those of this week. |
| Cybersecurity | An attacker adds the complaint cough to every form. Count how many suggestions of the test patients change, and which explanation would reveal the attack. |
| Health informatics | Find the threshold at which the team of the assistant and a nurse with an accuracy of 0.85 is most accurate, and the share of patients that the nurse then sees. |

## After the Colab notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-13/lab.html), tabs *Check yourself* and *Reflect and export*. The weekly task and the research assignment are on the [week overview](https://github.com/utkukose/human-computer-interaction-NB-lecture/tree/main/weeks/week-13).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>